In [ ]:
# ============================================================
# EDA
# ============================================================

# Install required libs if needed
#import sys
#!{sys.executable} -m pip install polars plotly seaborn matplotlib ydata-profiling --quiet

# Imports

from IPython.display import display

import itables
from itables import init_notebook_mode, show
init_notebook_mode( all_interactive = True )

import polars as pl
import polars.selectors as sc

import pandas as pd

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Set globally

# Display all columns and rows
pl.Config.set_tbl_cols( -1 )
pl.Config.set_tbl_rows( -1 )

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Set the maximum table width in characters
pl.Config.set_tbl_width_chars( -1 )

# Set the maximum length for strings in the display
pl.Config.set_fmt_str_lengths( 100 )


pl.Config( float_precision = 2 )

In [ ]:
def remove_outliers_iqr( df, numeric_cols ) :
  bounds = { }
  # Compute IQR bounds for each column
  for col in numeric_cols :
    stats = df.select( [
        pl.col( col ).quantile( 0.01 ).alias( "q1" ),
        pl.col( col ).quantile( 0.99 ).alias( "q3" )
    ] ).to_dict( as_series = False )

    q1 = stats[ "q1" ][ 0 ]
    q3 = stats[ "q3" ][ 0 ]
    
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    
    upper = q3 + 1.5 * iqr
    bounds[ col ] = ( lower, upper )

  # Build filter for all numeric columns
  filter_expr = None
  for col, ( low, high ) in bounds.items():
    expr = ( pl.col( col ) >= low ) & ( pl.col( col ) <= high )
    filter_expr = expr if filter_expr is None else ( filter_expr & expr )

  # Return filtered dataframe
  return df.filter( filter_expr )

In [ ]:
# ============================================================
# 1. LOAD PARQUET
# ============================================================

df = pl.read_parquet( 'data.parquet' )

# ============================================================
# 2. COLUMN GROUPING
# ============================================================

categorical_cols = [
    'COMPANY', 'TYPE', 'NAME',
    'BUCKET', 'BASE', 'GROUP', 'UN',
    'OIL_RIG'
]

# SERVICE order sometimes float/int — keep numeric
numeric_cols = [
    'ID', 'SERVICE',
    'ARQ_QTD', 'ARQ_TAM',
    'FALTA_QTD', 'FALTA_TAM',
    'FIELD', 'DEPTH'
]

date_cols = [
    'OLDEST', 'NEWEST', 'DATE'
]

files_cols = [
    'BUCKET', 'BASE',
    'ARQ_QTD', 'ARQ_TAM',
    'FALTA_QTD', 'FALTA_TAM',
]

service_cols = [
    'COMPANY', 'TYPE', 'NAME',
    'GROUP', 'UN', 'FIELD',
    'OIL_RIG', 'DEPTH',
]

# --------------------------
#
# --------------------------

df = df.with_columns(
    ( pl.col( 'ARQ_TAM' ) / 1024**2 ).round( decimals = 2 ),
    ( pl.col( 'FALTA_TAM' ) / 1024**2 ).round( decimals = 2 )
)



#df = remove_outliers_iqr( df, numeric_cols )

# ============================================================
# 3. ENSURE DATE COLUMNS ARE PARSED
# ============================================================

for col in date_cols :
    df = df.with_columns(
        pl.col( col ).cast( pl.Date )
    )

# ============================================================
# 4. ENCODE CATEGORICAL COLUMNS
# ============================================================

df = df.with_columns( [
    pl.col( col ).cast( pl.Categorical ) for col in categorical_cols
] )

df_encoded = df.with_columns([
    pl.col( col ).to_physical( ).alias( col + '_ENC' ) for col in categorical_cols
] )


show( df,
  layout = {
      'topStart'    : 'pageLength',
      'topEnd'      : 'paging',
      'bottomStart' : 'searchPanes',
      'bottomEnd'   : '',
  },
  searchPanes = {
    'layout'       : 'columns-4',
    'cascadePanes' : True
  }
)

In [ ]:
# ============================================================
# 5. TEXT EDA
# ============================================================

# --------------------------
# Basic info
# --------------------------
print("\n=== BASIC INFORMATION ===")
print("Shape:"), display( df.shape)

for c, t in zip(df.columns, df.dtypes):
    print(f"{c:<20}  {t}")



In [ ]:
# --------------------------
# NUMERIC SUMMARY
# --------------------------
print("\n=== NUMERIC SUMMARY ===")

#
show(
  df.select( numeric_cols ).drop( [ 'ID', 'SERVICE' ] ).describe(
    percentiles=[0.1, 0.3, 0.5, 0.7, 0.9],
    interpolation="linear",
  ).with_columns(
    sc.float( ).round( 2 )
  ),
  paging = False
)

# --------------------------
# Missing values
# --------------------------
print( "\n=== NULL COUNTS ===" )
show( df.null_count( ) )



In [ ]:
# --------------------------
# Top categories
# --------------------------
print( "\n=== TOP 10 CATEGORIES ===" )
for col in categorical_cols :
    print( f"\nTop 10 for {col}:" )
    show(
        df.group_by( col )
          .len( )
          .sort( "len", descending = True )
          .head( 10 )
    )


In [ ]:
# ============================================================
# 6. CORRELATION MATRIX (Numeric + Encoded Categorical)
# ============================================================

numeric_for_corr = numeric_cols + [c + "_ENC" for c in categorical_cols]
pdf_corr = df_encoded.select(numeric_for_corr).to_pandas()
corr_matrix = pdf_corr.corr()

print("\nCorrelation matrix:")
show(corr_matrix)



In [ ]:
# ============================================================
# 7. VISUAL EDA
# ============================================================

pdf = df_encoded.to_pandas()

In [ ]:
# Histograms
for col in numeric_cols:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()


In [ ]:
# Boxplots
for col in numeric_cols:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

In [ ]:

# Correlation Heatmap
plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Heatmap (with Encoded Categories)")
plt.show()


In [ ]:

# Category frequencies
for col in categorical_cols:
    top10 = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
          .to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10 {col}")
    fig.show()



In [ ]:
# ============================================================
# 8. PROFILING REPORT
# ============================================================

#from ydata_profiling import ProfileReport
#profile = ProfileReport(pdf, title="Dataset Profiling Report", explorative=True)
#profile.to_file("profiling_report.html")

print("\n✔ EDA Complete — profiling_report.html generated.")